# Step by step walk through

In [2]:
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import pandas as pd
import re
load_dotenv()
import docx 
import nltk
nltk.download('punkt_tab')
import time
from tqdm import tqdm
from pinecone import Pinecone

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\steven\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
C:\Users\steven\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


# Test out OpenAI

In [3]:
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key = os.environ.get("OPENAI_API_KEY"))


def chat_with_openai(prompt):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role":"system", "content": "You are a financial expert and be helpful"},
                  {"role": "user", "content":prompt}],
                  max_tokens = 100
    )
    return response.choices[0].message.content

In [4]:
prompt = "what is openAi"

In [5]:
bot = chat_with_openai(prompt)
print(bot)

OpenAI is an artificial intelligence research laboratory consisting of both a for-profit company and a non-profit organization. The goal of OpenAI is to ensure that artificial general intelligence (AGI) benefits all of humanity. They conduct research in various areas of artificial intelligence, from computer vision to natural language processing. OpenAI also aims to promote and develop AI technology in an ethical and safe manner.


### Good to understand how to call endpoints:
https://github.com/openai/openai-python/discussions/742

# Let's clean our investment restrictions data to generalise these and use them to prompt engineer the LLM

In [6]:
# load the excel file which contains our investment restrictions
file_path = "Synethic LDI restrictions.xlsx"
df2 = pd.read_excel(file_path)

In [7]:
# use regex to standardize the limits to X% and £X
def replace_limits(text: str) -> str:
    text = re.sub(r'\b\d+(\.\d+)?%', 'X%', text) # replace % limits
    text = re.sub(r'£\s?\d+([.,]?\d+)*\s?[MBK]?', '£X', text) # replace £ limits
    return text 

In [8]:
#Create a new column in our dataframe applying the replace_limits function
df2["General mandate transcript"] = df2["Mandate transcript"].apply(replace_limits)

In [9]:
df2

,Rule ID,Mandate transcript,General mandate transcript
0,87897,The cumulative PV01 of the benchmark at each t...,The cumulative PV01 of the benchmark at each t...
1,43203,The duration of liabilities should match withi...,The duration of liabilities should match withi...
2,22329,The Bond PV01 of the portfolio shall not devia...,The Bond PV01 of the portfolio shall not devia...
3,5507,The cumulative PV01 of the benchmark at each t...,The cumulative PV01 of the benchmark at each t...
4,51691,Asset allocation should not deviate by more th...,Asset allocation should not deviate by more th...
...,...,...,...
176,45066,The duration of liabilities should match withi...,The duration of liabilities should match withi...
177,78149,Asset allocation should not deviate by more th...,Asset allocation should not deviate by more th...
178,24461,The PV01 of the portfolio shall not deviate by...,The PV01 of the portfolio shall not deviate by...
179,1763,The PV01 of the portfolio shall not deviate by...,The PV01 of the portfolio shall not deviate by...


In [10]:
# save the file path
output_path = "Generic investment restrictions.xlsx"
df2.to_excel(output_path,index=False)

In [11]:
# Load the word document and save to doc
doc_path = "guidelines.docx"
doc = docx.Document(doc_path)

## The below code is how we will pre process the document by using NLTK to tokenize our paragraphs into sentences
We will follow the format {paragraph index: {sentence index: text}} to keep track of the location of each sentence which will help us annotate the word document later!

In [12]:
#create a dictionary to store dic = {para_index: {sentence_index:text}}
para_dic = {}

# iterate over the index and paragraphs
for index,paragraph in enumerate(doc.paragraphs):
    paragraph_text = paragraph.text.strip() # split the data into paragraphs and remove whitespaces

    # use nltk to tokenize your paragraphs into sentences!
    sentences = nltk.sent_tokenize(paragraph_text)

    # initialise a sentence dic
    sentence_dic = {}

    # then you need to remember to keep track of the index of the paragraph and sentence!
    for i, sentence in enumerate(sentences):
        sentence = re.sub(r'[^\x00-\x7F]+', '', sentence)  # Remove non-ASCII characters
        sentence = re.sub(r'^\d+\.\s*', '', sentence)  # Remove number bullet points at the start (e.g 3.)
        sentence = re.sub(r'\s+', ' ', sentence).strip()  # Remove extra spaces

        # check if the sentence is not empty and contains more than just numbers (becareful as restrictions may come in tables with numbers)
        if bool(sentence) and not re.match(r'^\d+\s*$', sentence):    
            sentence_dic[i] = sentence
    
    # store the paragraph index : {sentence index : text}
    para_dic[index]= sentence_dic


In [13]:
# remove keys where the sentence has an empty dictionary => {sentence index: text} is empty
para_dic =  {k:v for k,v in para_dic.items() if v}
para_dic

{3: {0: 'Overview'},
 4: {0: 'The Guidance for Developing an Investment Policy Statement, adopted by the Pension Review Board (PRB), provides a description of policies and sections that systems are encouraged to include in their investment policy statement (IPS), as applicable.',
  1: 'This IPS example is an additional reference tool provided by the PRB to demonstrate what each type of policy might look like in an IPS.',
  2: 'Specific requirements included below (including defined roles and responsibilities, defined percentages, etc.)',
  3: 'are not intended to be prescriptive but to help inform users about the types of elements that should be included in an Investment Policy.',
  4: 'In addition,Tthis document is not intended to be a fully functioning IPS since certain policy sections have been shortened for brevity and each IPS should be tailored to each systems needs.',
  5: 'Furthermore, this IPS example is not intended to replace any systems existing IPS, but systems may use it 

# Now that we have a dictionary of sentences, we need to ask the LLM to give us a subset of this dictionary by definining which ones are investment restrictions and store them in a restriction dictionary

# Next we will begin to prompt engineer our LLM by giving it examples of investment restrictions and general texts

Below is an example using 1 sentence

In [14]:
# Test with one example
sentence_text = "GFOA Best Practice, Investment Policies for Defined Benefit Plans (Sept. 30, 2017)."

# we used the generalised investment restrictions in the prompt as examples
prompt = f"""
    You are a financial expert on investment restrictions and guidelines. Your task is to determine whether a given sentence is an 'investment restriction' or 'general text'. Below are some examples:
    Examples of investment restrictions:
    1.	The cumulative PV01 of the benchmark at each tenor should not deviate by more than X% than the total benchmark 
    2.	The duration of liabilities should match within X% of the benchmark liabilities duration
    3.	The Bond PV01 of the portfolio shall not deviate by more than X% of the benchmark
    4.	Asset allocation should not deviate by more than X% in any asset class
    5.	The Swap PV01 of the portfolio shall not deviate by more than X% of the benchmark
    6.	Cash holdings should not exceed X% of the total portfolio
    7.	Permitted assets: UK Gilts, Repo, Reverse repo
    8.	The PV01 of the portfolio shall not deviate by more than X% of the benchmark
    9.	Portfolio duration should be within X% of the benchmark duration
    10.	Derivatives exposure should not exceed X% of the portfolio value

    Examples of general text:
    1. This document outlines the investment policy for the fiscal year.
    2. The portfolio manager is responsible for executing trades based on the client's needs.


    Now classify the following sentences:
        
    sentence: {sentence_text}

    Classify the sentence as either 'investment restriction' or 'general text'.
        """

response= client.completions.create(
        model = "gpt-3.5-turbo-instruct",
        prompt = prompt,
        max_tokens = 10,
        temperature=0
    )

classifier = response.choices[0].text.strip().lower()

if classifier == "investment restriction":
    print(f'This is an investment restriction {classifier}')
else:
    print(f'not an investment restriction: {classifier}')

not an investment restriction: general text


# We used a multi turn chat to get the desired output from the LLM.

System - Controls the assistants behaviour

User - Instructs the assistant

Assistant - Responds to the user instruction

The model will learn from these few shot examples without additional training.

In [15]:
# Defining the model
def classify_sentence(sentence):

    system_prompt = """You are a financial expert in investment guidelines. You will classify each sentence in triple backticks given to you as either 'investment restriction' or 'general text'. I have provided
    some examples of sentences with there classification. These have been generalised where the X% or £X can represent any numerical value."""

    response = client.chat.completions.create(
        model = "gpt-3.5-turbo",
        messages=[{"role":"system", "content": system_prompt},
                  {"role": "user", "content": "sentence: ```The cumulative PV01 of the benchmark at each tenor should not deviate by more than X% than the total benchmark ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```The duration of liabilities should match within X% of the benchmark liabilities duration ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```The Bond PV01 of the portfolio shall not deviate by more than X% of the benchmark ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```This document outlines the investment policy for the fiscal year. ``` " },
                  {"role": "assistant", "content": "general text"},
                  {"role": "user", "content": "sentence: ```Asset allocation should not deviate by more than X% in any asset class ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```The Swap PV01 of the portfolio shall not deviate by more than X% of the benchmark``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```Cash holdings should not exceed X% of the total portfolio ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```The portfolio manager is responsible for executing trades based on the client's needs. ``` " },
                  {"role": "assistant", "content": "general text"},
                  {"role": "user", "content": "sentence: ```Permitted assets: UK Gilts, Repo, Reverse repo ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```The PV01 of the portfolio shall not deviate by more than X% of the benchmark ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```Portfolio duration should be within X% of the benchmark duration ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": "sentence: ```Derivatives exposure should not exceed X% of the portfolio value ``` " },
                  {"role": "assistant", "content": "investment restriction"},
                  {"role": "user", "content": f"sentence: ```{sentence}```"}],
                  temperature=0,
                  max_tokens=10
    )
    return response.choices[0].message.content.strip().lower()

# Below we can see that the LLM took almost 4 minutes to analyze each sentence and classify whether it's an investment restriction or not and stored them in the res dictionary

In [16]:
start_time = time.time()
res = {} # dictionary to store the restrictions

# iterate over the paragraphs dictionary
for i,v in tqdm(para_dic.items(), desc="Processing paragraphs"):
    temp2 = {}
    # For each paragraph, iterate over it's sentences
    for ide, sen in v.items():
        result = classify_sentence(sen) # input each sentence into the LLM for classification
        print(f'Sentence: {sen} | Classification: {result}')
        if result == "investment restriction":
            temp2[ide]= sen # only add sentences which are classified as investment restrictions
    res[i]= temp2

# clean up the restriction dictionary
res = {ke:va for ke,va in res.items() if va}

end_time = time.time()
elapsed_time = end_time - start_time
print(f'Time taken to classify: {elapsed_time: .4f} seconds')

Processing paragraphs:   0%|          | 1/211 [00:00<01:20,  2.62it/s]

Sentence: Overview | Classification: general text
Sentence: The Guidance for Developing an Investment Policy Statement, adopted by the Pension Review Board (PRB), provides a description of policies and sections that systems are encouraged to include in their investment policy statement (IPS), as applicable. | Classification: general text
Sentence: This IPS example is an additional reference tool provided by the PRB to demonstrate what each type of policy might look like in an IPS. | Classification: general text
Sentence: Specific requirements included below (including defined roles and responsibilities, defined percentages, etc.) | Classification: general text
Sentence: are not intended to be prescriptive but to help inform users about the types of elements that should be included in an Investment Policy. | Classification: general text
Sentence: In addition,Tthis document is not intended to be a fully functioning IPS since certain policy sections have been shortened for brevity and eac

Processing paragraphs:   1%|          | 2/211 [00:02<05:09,  1.48s/it]

Sentence: Furthermore, this IPS example is not intended to replace any systems existing IPS, but systems may use it as a starting point of a new IPS or to develop new policy language to update an existing policy. | Classification: general text
Sentence: This document contains example language from industry entities such as the Government Finance Officers Association (GFOA) and the Chartered Financial Analyst Institute (CFAI). | Classification: general text
Sentence: Specific references are provided at the end of this document. | Classification: general text
Sentence: The PRB also used policy language from actual IPS documents adopted by several Texas public retirement systems including, but not limited to, the Texas Municipal Retirement System (TMRS), Texas County and District Retirement System (TCDRS), Teacher Retirement System of Texas (TRS), City of Austin Employees Retirement System (COAERS), Irving Firemens Relief and Retirement Fund, Fort Worth Employees Retirement Fund, City of 

Processing paragraphs:   1%|▏         | 3/211 [00:04<05:46,  1.67s/it]

...

Sentence: https://www.gfoa.org/materials/adopting-financial-policies | Classification: general text


Processing paragraphs: 100%|█████████▉| 210/211 [02:08<00:00,  1.70it/s]

Sentence: - GFOA Best Practice, Investment Policies for Defined Benefit Plans (Sept. 30, 2017). | Classification: general text
Sentence: https://www.gfoa.org/materials/investment-policies-for-defined-benefit-plans - GFOA Best Practice, Investment Fee Guidelines for External Management of Defined Benefit Plans (Sept. 28. | Classification: general text
Sentence: 2018). | Classification: general text


Processing paragraphs: 100%|██████████| 211/211 [02:09<00:00,  1.63it/s]

Sentence: https://www.gfoa.org/materials/investment-fee-guidelines - GFOA Alternative Investments Checklist - GFOA Sample Investment Policy | Classification: general text
Time taken to classify:  129.8258 seconds


# The res dictionary contains the output of the LLM classification which holds all of the sentences it classified as investment restrictions!

# Now we need to vectorize the sentences in the res dictionary to be able query pinecone and retrieve the rule ID which is the closest match to the input vector within the database

We have used Cosine Similarity which calculates the Cosine of the angle between two vectors. The similarity ranges from -1 to 1

1 - The vectors point in the same direction (maximum similarity)

0 - The vectors are orthogonal (at right angles to each other), meaning they have no similarity

-1 - The vectors point in opposite directions (maximum dissimilarity)

# To vectorize our sentences we need to embed them by using the text embeding model from openAI

In [17]:
# call open ai to embedd your data
def get_embedding(text_to_embed):
    response = client.embeddings.create(
        model = "text-embedding-3-small",
        input = [text_to_embed]
    )
    embedding = response.data[0].embedding
    return embedding

In [18]:
vectorized_dict = {}

for inde,valu in tqdm(res.items(),desc="Vectorizing"):
    temp3 = {}
    for index3,values3 in valu.items():
        vector = get_embedding(f'"{values3}"') # vectorize each sentence from the res dictionary
        temp3[index3]=vector
    vectorized_dict[inde]=temp3

Vectorizing: 100%|██████████| 31/31 [00:10<00:00,  3.00it/s]


In [19]:
vectorized_dict

{33: {0: [-0.0026626286562532187,
   0.004591410979628563,
   0.10234884917736053,
   -0.015118535608053207,
   -0.011410337872803211,
   0.030730631202459335,
   0.04844685271382332,
   -0.0074293832294642925,
   0.019326787441968918,
   0.016716111451387405,
   -0.003909518010914326,
   -0.012566308490931988,
   -0.03605588898062706,
   -0.013806703500449657,
   0.06634490936994553,
   0.052577175199985504,
   0.003055528737604618,
   -0.007624209858477116,
   -0.026262609288096428,
   -0.006257177330553532,
   0.005263562314212322,
   0.010040057823061943,
   -0.017352545633912086,
   0.047277893871068954,
   0.01797598972916603,
   -0.06514997780323029,
   -0.04234229028224945,
   -0.020093103870749474,
   0.06520193070173264,
   -0.003562077647075057,
   0.03740668669342995,
   -0.013365096412599087,
   0.003034422406926751,
   -0.02644444815814495,
   -0.020664595067501068,
   0.04914822801947594,
   0.021417925134301186,
   0.02124907448887825,
   -0.020794479176402092,
   -0.01

# Time to query pinecone to retrieve the rule IDs and store it in a dictionary: {paragraph index: {sentence index: rule ID}}

Vector - the vectors we want to match

top k - return the top k similar results

Include_values - The actual vectors from the vector database

include_metadata - This would be the mandate transcript

In [20]:
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY"))
index = pc.Index("guideline-bot")

In [21]:
# Retrieve the rule IDs
rules_dict = {}
for index4, values4 in tqdm(vectorized_dict.items(), desc="Searching similar rules"):
    temp4 = {}
    for index5, values5 in values4.items():
        query_response = index.query(vector=values5, top_k=1, include_values=False, include_metadata=False)
        print(query_response['matches'][0]['score'])
        temp4[index5] = query_response['matches'][0]['id']
    rules_dict[index4] = temp4

Searching similar rules:   6%|▋         | 2/31 [00:01<00:12,  2.24it/s]

0.965476155
0.975057423


Searching similar rules:  13%|█▎        | 4/31 [00:01<00:06,  4.47it/s]

0.972847104
0.86207664


Searching similar rules:  19%|█▉        | 6/31 [00:01<00:04,  6.15it/s]

...

0.464866608
0.591144323


Searching similar rules:  94%|█████████▎| 29/31 [00:04<00:00,  8.00it/s]

0.391024679
0.505383074


Searching similar rules: 100%|██████████| 31/31 [00:04<00:00,  6.81it/s]

0.382778
0.46710518


In [22]:
rules_dict

{33: {0: '87897'},
 34: {0: '22329'},
 35: {0: '43203'},
 38: {0: '97771'},
 39: {0: '72440'},
 40: {0: '34225'},
 41: {0: '12896'},
 42: {0: '68722'},
 43: {0: '69079'},
 45: {0: '8299'},
 46: {0: '26909'},
 47: {0: '64569'},
 48: {0: '35943'},
 49: {0: '45281'},
 50: {0: '70367'},
 52: {0: '84709'},
 78: {0: '93356'},
 79: {0: '93356'},
 80: {1: '77000'},
 81: {1: '62622'},
 82: {1: '28087'},
 88: {0: '25058'},
 93: {0: '25058'},
 117: {0: '12896'},
 118: {0: '73364', 1: '73364'},
 124: {0: '12896'},
 125: {0: '73364'},
 126: {0: '15477'},
 128: {0: '12896'},
 129: {0: '12896'},
 130: {0: '73364'}}

In [23]:
res

{33: {0: 'The cumulative PV01 of the benchmark at each tenor should not deviate by more than 10% than the total benchmark mark'},
 34: {0: 'The Bond PV01 of the portfolio shall not deviate by more than 14% of the benchmark'},
 35: {0: 'The duration of liabilities should match within 14% of the benchmark liabilities duration'},
 38: {0: 'Cash holdings should not exceed 4% of the total portfolio.'},
 39: {0: 'The Bond PV01 of the portfolio shall not deviate by more than 18% of the benchmark.'},
 40: {0: 'The Bond PV01 of the portfolio shall not deviate by more than 1% of the benchmark.'},
 41: {0: 'Permitted assets: UK Gilts, Repo, Reverse repo.'},
 42: {0: 'The Bond PV01 of the portfolio shall not deviate by more than 11% of the benchmark.'},
 43: {0: 'Cash holdings should not exceed 2% of the total portfolio.'},
 45: {0: 'The cumulative PV01 of the benchmark at each tenor should not deviate by more than 2% than the total benchmark mark.'},
 46: {0: 'Asset allocation should not deviate 

# Below is the code which will allow us to annotate each sentence with it's rule ID

Note - check out the full code in the app4.py as the rules_dict in this note book contains paragraphs which contains tables. The issue here is if we include those paragraphs with tables then the subsequent paragraphs which we need to annotate will have there indexes misalligned because of the presence of the table in the previous paragraph. As a work around, I did not include any paragraph which contained a table in my app4.py file.

In [25]:
from win32com import client as win32
word_app = win32.Dispatch("Word.Application")
word_app.Visible=True
doc3 = word_app.Documents.Open(r'C:\Users\steven\Desktop\Python\guidelines.docx')

for para_index, sentences in rules_dict.items():
    para = doc3.Paragraphs(para_index+1) # retrieves the line I need to annotate
    start= para.Range.Start
    end = para.Range.End
    sen_range = doc3.Range(start,end)
    for sen_index,rules in sentences.items():
        doc3.Comments.Add(sen_range,f'Rule ID: {rules}')

doc3.SaveAs2(r'C:\Users\steven\Desktop\Python\guidelinesV8.docx') # Change to your desired output path
doc3.Close()
word_app.Quit()

com_error: (-2147352567, 'Exception occurred.', (0, 'Microsoft Word', 'This method or property is not available because the object refers to the end of a table row.', 'wdmain11.chm', 37373, -2146823683), None)